# Flujo: siguientes experimentos NLP
Use un runtime con GPU. El paquete contiene código y corpus locales; las etiquetas son heredadas y requieren revisión. Los casos de diagnóstico no se incluyen en entrenamiento. Los resultados nuevos no reemplazan automáticamente al A100 actual.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
root = Path('/content/flujo_hybrid_v2')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (root/name).resolve().is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(root)
os.chdir(root)


In [ ]:
%pip install -q -r requirements-hybrid-colab-v2.txt
import torch
assert torch.cuda.is_available(), "Seleccione un runtime GPU"
print(torch.cuda.get_device_name(0))


## Comparaciones controladas
Las semillas usan la misma partición original. Cada ablación cambia un parámetro frente al A100 original: contexto, capas o épocas. El experimento ampliado es otra comparación y no aísla cambios de arquitectura.

In [ ]:
!python scripts/run_hybrid_experiments.py


In [ ]:
!python scripts/run_hybrid_experiments.py --run seed7 seed21 context128 layers2 epoch1 expanded


## Guardar resultados
Descargue los reportes para revisar antes de elegir un nuevo modelo. Los modelos quedan en artifacts/hybrid/experiments-v2; descárguelos antes de cerrar el runtime si desea conservarlos.

In [ ]:
import shutil
shutil.make_archive('/content/flujo_experiment_reports_v2', 'zip', 'reports')
files.download('/content/flujo_experiment_reports_v2.zip')


In [ ]:
# Para exportar un candidato después de revisar sus resultados, elija su ID.
model_to_export = None  # por ejemplo 'expanded'; no implica que sea mejor
if model_to_export is not None:
    directory = Path('artifacts/hybrid/experiments-v2') / model_to_export
    assert directory.is_dir() and directory.parent == Path('artifacts/hybrid/experiments-v2')
    output = shutil.make_archive('/content/flujo_model_' + model_to_export, 'zip', directory)
    files.download(output)
